# Notebook 3 — Repeatability Validation

**Scientific question:** Is the measured $\tau_{\text{BP}}$ stable across independent random initializations / seeds?


### Cell 1 — Imports and configuration


In [ ]:
# ============================================================
# NOTEBOOK 3 — REPEATABILITY VALIDATION
# ============================================================
#
# Scientific question:
#
#   Is the measured tau_BP stable across independent random
#   initializations / seeds?
#
# IMPORTANT:
#   - Prefer existing seed-level data.
#   - Do NOT rerun quantum simulations if suitable seed-level
#     data already exist.
#   - Do NOT hardcode previous tau values.
#   - Do NOT hardcode previous repeatability statistics.
#   - Raw files are never modified.
#
# ============================================================

import os
import glob
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

OUTPUT_DIR = "notebook3_repeatability"
os.makedirs(OUTPUT_DIR, exist_ok=True)

THRESHOLD = 1e-2

print("=" * 80)
print("NOTEBOOK 3 — REPEATABILITY VALIDATION")
print("=" * 80)
print("Quantum simulation will only be considered if seed-level")
print("repeatability data cannot be reconstructed from existing files.")

### Cell 2 — Search for existing repeatability data


In [ ]:
# ============================================================
# CELL 2 — SEARCH FOR EXISTING REPEATABILITY DATA
# ============================================================

SEARCH_ROOTS = [
    ".",
    "/content",
]

candidate_files = []

for root in SEARCH_ROOTS:
    if not os.path.exists(root):
        continue

    for pattern in ["*.csv", "*.pkl", "*.pickle", "*.json"]:
        candidate_files.extend(
            glob.glob(
                os.path.join(root, "**", pattern),
                recursive=True
            )
        )

candidate_files = sorted(set(candidate_files))

keywords = [
    "repeat",
    "seed",
    "stability",
    "variability",
    "variance",
    "gradient",
    "tau",
]

repeatability_candidates = []

for path in candidate_files:
    name = os.path.basename(path).lower()

    # Ignore current output directory
    if OUTPUT_DIR in path.replace("\\", "/"):
        continue

    if any(k in name for k in keywords):
        repeatability_candidates.append(path)

print("=" * 80)
print("POTENTIAL REPEATABILITY DATASETS")
print("=" * 80)

if repeatability_candidates:
    for p in repeatability_candidates:
        print(p)
else:
    print("No candidate repeatability files found.")

### Cell 3 — Inspect CSV schemas


In [ ]:
# ============================================================
# CELL 3 — INSPECT CSV SCHEMAS
# ============================================================

schema_rows = []

for path in repeatability_candidates:

    if not path.lower().endswith(".csv"):
        continue

    try:
        df = pd.read_csv(path)

        schema_rows.append({
            "file": path,
            "rows": len(df),
            "columns": len(df.columns),
            "columns_list": ", ".join(map(str, df.columns)),
        })

    except Exception as e:

        schema_rows.append({
            "file": path,
            "rows": np.nan,
            "columns": np.nan,
            "columns_list": f"ERROR: {e}",
        })

schema_df = pd.DataFrame(schema_rows)

if len(schema_df) > 0:
    display(schema_df)
else:
    print("No readable candidate CSVs found.")

### Cell 4 — Identify seed-level datasets


In [ ]:
# ============================================================
# CELL 4 — IDENTIFY SEED-LEVEL DATA
# ============================================================

def find_column(columns, candidates):
    """
    Return the actual column name matching one of the candidate
    lowercase names.
    """
    lookup = {
        str(c).lower(): c
        for c in columns
    }

    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]

    return None


seed_level_candidates = []

for path in repeatability_candidates:

    if not path.lower().endswith(".csv"):
        continue

    try:
        df = pd.read_csv(path)

        n_col = find_column(df.columns, ["n"])
        k_col = find_column(df.columns, ["k"])
        seed_col = find_column(df.columns, ["seed", "random_seed"])
        tau_col = find_column(
            df.columns,
            ["tau_BP", "tau", "tau_bp"]
        )

        depth_col = find_column(
            df.columns,
            ["depth", "L", "circuit_depth"]
        )

        var_col = find_column(
            df.columns,
            ["var_pooled", "gradient_variance", "variance"]
        )

        if (
            n_col is not None
            and k_col is not None
            and seed_col is not None
            and (
                tau_col is not None
                or (
                    depth_col is not None
                    and var_col is not None
                )
            )
        ):
            seed_level_candidates.append({
                "path": path,
                "n_col": n_col,
                "k_col": k_col,
                "seed_col": seed_col,
                "tau_col": tau_col,
                "depth_col": depth_col,
                "var_col": var_col,
            })

    except Exception:
        pass

print("=" * 80)
print("SEED-LEVEL DATASETS")
print("=" * 80)

if seed_level_candidates:

    for item in seed_level_candidates:
        print(item)

else:

    print(
        "No dataset containing sufficient seed-level information "
        "was identified."
    )

### Cell 5 — Load existing seed-level data


In [ ]:
# ============================================================
# CELL 5 — LOAD EXISTING SEED-LEVEL DATA
# ============================================================

seed_data = None
seed_source = None
seed_schema = None

if seed_level_candidates:

    seed_schema = seed_level_candidates[0]
    seed_source = seed_schema["path"]

    seed_data = pd.read_csv(seed_source)

    print("=" * 80)
    print("SELECTED REPEATABILITY DATASET")
    print("=" * 80)

    print(f"File: {seed_source}")
    print(f"Rows: {len(seed_data):,}")
    print(
        f"Columns: {seed_data.columns.tolist()}"
    )

else:

    print(
        "No suitable seed-level dataset is available in the "
        "current working directory."
    )

### Cell 6 — Standardize seed-level data


In [ ]:
# ============================================================
# CELL 6 — STANDARDIZE SEED DATA
# ============================================================

if seed_data is None:

    print("=" * 80)
    print("REPEATABILITY DATA STATUS")
    print("=" * 80)
    print(
        "STATUS: INCOMPLETE — suitable seed-level data were not found."
    )
    print(
        "No quantum simulation has been started automatically."
    )
    print(
        "A fresh repeatability simulation is required before "
        "Checkpoint 3 can be completed."
    )

else:

    n_col = seed_schema["n_col"]
    k_col = seed_schema["k_col"]
    seed_col = seed_schema["seed_col"]
    tau_col = seed_schema["tau_col"]
    depth_col = seed_schema["depth_col"]
    var_col = seed_schema["var_col"]

    repeat_df = pd.DataFrame({
        "n": pd.to_numeric(
            seed_data[n_col],
            errors="coerce"
        ),
        "k": pd.to_numeric(
            seed_data[k_col],
            errors="coerce"
        ),
        "seed": seed_data[seed_col],
    })

    if tau_col is not None:

        repeat_df["tau_BP"] = pd.to_numeric(
            seed_data[tau_col],
            errors="coerce"
        )

    else:

        repeat_df["depth"] = pd.to_numeric(
            seed_data[depth_col],
            errors="coerce"
        )

        repeat_df["var_pooled"] = pd.to_numeric(
            seed_data[var_col],
            errors="coerce"
        )

    repeat_df = repeat_df.dropna(
        subset=["n", "k", "seed"]
    )

    repeat_df["n"] = repeat_df["n"].astype(int)
    repeat_df["k"] = repeat_df["k"].astype(int)

    print("=" * 80)
    print("STANDARDIZED REPEATABILITY DATA")
    print("=" * 80)

    display(
        repeat_df.head(20)
    )

### Cell 7 — Derive $\tau_{\text{BP}}$ from seed-level variance curves


In [ ]:
# ============================================================
# CELL 7 — DERIVE TAU FROM SEED-LEVEL VARIANCE CURVES
# ============================================================

if seed_data is not None:

    if "tau_BP" not in repeat_df.columns:

        print(
            "tau_BP not directly stored. "
            "Deriving it from each seed's variance curve."
        )

        tau_records = []

        for (n, k, seed), group in repeat_df.groupby(
            ["n", "k", "seed"]
        ):

            group = group.sort_values("depth")

            crossed = group[
                group["var_pooled"] < THRESHOLD
            ]

            if len(crossed) > 0:

                tau = int(
                    crossed.iloc[0]["depth"]
                )

                censored = False

            else:

                max_depth = int(
                    group["depth"].max()
                )

                tau = max_depth + 1
                censored = True

            tau_records.append({
                "n": int(n),
                "k": int(k),
                "seed": seed,
                "tau_BP": tau,
                "censored": censored,
            })

        tau_seed_df = pd.DataFrame(
            tau_records
        )

    else:

        tau_seed_df = repeat_df[
            ["n", "k", "seed", "tau_BP"]
        ].copy()

        tau_seed_df["censored"] = False

        # If the original file explicitly contains a censor flag,
        # retain it.
        for possible_name in [
            "censored",
            "right_censored"
        ]:
            if possible_name in repeat_df.columns:
                tau_seed_df["censored"] = (
                    repeat_df[possible_name].astype(bool)
                )
                break

    tau_seed_df = tau_seed_df.sort_values(
        ["n", "k", "seed"]
    ).reset_index(drop=True)

    print("=" * 80)
    print("TAU PER SEED")
    print("=" * 80)

    display(tau_seed_df)

### Cell 8 — Seed and configuration coverage


In [ ]:
# ============================================================
# CELL 8 — SEED / CONFIGURATION COVERAGE
# ============================================================

if seed_data is not None:

    seed_counts = (
        tau_seed_df
        .groupby(["n", "k"])["seed"]
        .nunique()
        .reset_index(
            name="n_seeds"
        )
    )

    print("=" * 80)
    print("SEED COVERAGE BY CONFIGURATION")
    print("=" * 80)

    display(seed_counts)

    print(
        f"\nTotal configurations: "
        f"{len(seed_counts)}"
    )

    print(
        f"Minimum seeds/configuration: "
        f"{seed_counts['n_seeds'].min()}"
    )

    print(
        f"Maximum seeds/configuration: "
        f"{seed_counts['n_seeds'].max()}"
    )

### Cell 9 — Repeatability statistics


In [ ]:
# ============================================================
# CELL 9 — REPEATABILITY STATISTICS
# ============================================================

if seed_data is not None:

    summary_rows = []

    for (n, k), group in tau_seed_df.groupby(
        ["n", "k"]
    ):

        values = group["tau_BP"].astype(float)

        mean_tau = float(values.mean())
        std_tau = float(values.std(ddof=1)) if len(values) > 1 else np.nan

        cv = (
            std_tau / mean_tau
            if len(values) > 1 and mean_tau != 0
            else np.nan
        )

        summary_rows.append({
            "n": int(n),
            "k": int(k),
            "n_seeds": int(values.size),
            "tau_mean": mean_tau,
            "tau_std": std_tau,
            "tau_CV": cv,
            "tau_min": float(values.min()),
            "tau_max": float(values.max()),
            "tau_range": float(values.max() - values.min()),
            "censored_seeds": int(
                group["censored"].sum()
            ),
        })

    repeatability_summary = pd.DataFrame(
        summary_rows
    ).sort_values(
        ["n", "k"]
    )

    print("=" * 80)
    print("REPEATABILITY SUMMARY")
    print("=" * 80)

    display(
        repeatability_summary
    )

### Cell 10 — Identify most and least stable cases


In [ ]:
# ============================================================
# CELL 10 — STABILITY EXTREMES
# ============================================================

if seed_data is not None:

    usable = repeatability_summary.dropna(
        subset=["tau_CV"]
    ).copy()

    if len(usable) > 0:

        most_stable = usable.sort_values(
            "tau_CV"
        ).iloc[0]

        least_stable = usable.sort_values(
            "tau_CV",
            ascending=False
        ).iloc[0]

        print("=" * 80)
        print("REPEATABILITY EXTREMES")
        print("=" * 80)

        print(
            "Most stable configuration:"
        )

        print(
            f"n={int(most_stable['n'])}, "
            f"k={int(most_stable['k'])}, "
            f"CV={100*most_stable['tau_CV']:.2f}%"
        )

        print()

        print(
            "Least stable configuration:"
        )

        print(
            f"n={int(least_stable['n'])}, "
            f"k={int(least_stable['k'])}, "
            f"CV={100*least_stable['tau_CV']:.2f}%"
        )

### Cell 11 — Select representative configurations automatically


In [ ]:
# ============================================================
# CELL 11 — REPRESENTATIVE CASES
# ============================================================

if seed_data is not None:

    representative_cases = []

    grouped = (
        repeatability_summary
        .sort_values(["n", "k"])
    )

    # Select:
    # 1. Smallest observed tau
    # 2. Median tau
    # 3. Largest observed tau

    smallest = (
        grouped.sort_values("tau_mean")
        .iloc[0]
    )

    largest = (
        grouped.sort_values(
            "tau_mean",
            ascending=False
        )
        .iloc[0]
    )

    median_idx = (
        grouped["tau_mean"]
        .sub(grouped["tau_mean"].median())
        .abs()
        .idxmin()
    )

    median_case = grouped.loc[
        median_idx
    ]

    for label, row in [
        ("early_onset", smallest),
        ("mid_onset", median_case),
        ("late_onset", largest),
    ]:

        representative_cases.append({
            "label": label,
            "n": int(row["n"]),
            "k": int(row["k"]),
        })

    representative_df = pd.DataFrame(
        representative_cases
    )

    print("=" * 80)
    print("AUTOMATICALLY SELECTED REPRESENTATIVE CASES")
    print("=" * 80)

    display(
        representative_df
    )

### Cell 12 — Plot $\tau_{\text{BP}}$ distribution across seeds


In [ ]:
# ============================================================
# CELL 12 — TAU DISTRIBUTION BY REPRESENTATIVE CASE
# ============================================================

if seed_data is not None:

    for _, case in representative_df.iterrows():

        n_val = int(case["n"])
        k_val = int(case["k"])
        label = case["label"]

        subset = tau_seed_df[
            (tau_seed_df["n"] == n_val)
            &
            (tau_seed_df["k"] == k_val)
        ]

        plt.figure(figsize=(8, 5))

        plt.hist(
            subset["tau_BP"],
            bins=max(
                3,
                int(
                    subset["tau_BP"].nunique()
                    + 1
                )
            ),
            edgecolor="black"
        )

        mean_tau = subset["tau_BP"].mean()

        plt.axvline(
            mean_tau,
            linestyle="--",
            linewidth=2,
            label=f"Mean = {mean_tau:.2f}"
        )

        plt.xlabel("tau_BP")
        plt.ylabel("Frequency")
        plt.title(
            f"Repeatability: n={n_val}, k={k_val} ({label})"
        )
        plt.grid(True, alpha=0.3)
        plt.legend()

        plt.tight_layout()

        plt.savefig(
            os.path.join(
                OUTPUT_DIR,
                f"tau_distribution_n{n_val}_k{k_val}.png"
            ),
            dpi=200
        )

        plt.show()

### Cell 13 — Final repeatability assessment


In [ ]:
# ============================================================
# CELL 13 — FINAL REPEATABILITY ASSESSMENT
# ============================================================

if seed_data is None:

    print("=" * 80)
    print("CHECKPOINT 3 STATUS")
    print("=" * 80)
    print(
        "INCOMPLETE — existing seed-level repeatability data "
        "could not be reconstructed."
    )
    print(
        "No simulation was started automatically."
    )

else:

    n_configs = len(repeatability_summary)

    finite_cv = repeatability_summary[
        np.isfinite(
            repeatability_summary["tau_CV"]
        )
    ]

    print("=" * 80)
    print("CHECKPOINT 3 — REPEATABILITY SUMMARY")
    print("=" * 80)

    print(
        f"Configurations analyzed: {n_configs}"
    )

    print(
        f"Minimum seeds/configuration: "
        f"{seed_counts['n_seeds'].min()}"
    )

    print(
        f"Maximum seeds/configuration: "
        f"{seed_counts['n_seeds'].max()}"
    )

    if len(finite_cv) > 0:

        print(
            f"Median tau CV: "
            f"{100 * finite_cv['tau_CV'].median():.2f}%"
        )

        print(
            f"Maximum tau CV: "
            f"{100 * finite_cv['tau_CV'].max():.2f}%"
        )

    print(
        f"Total censored seed observations: "
        f"{int(tau_seed_df['censored'].sum())}"
    )

    print()
    print(
        "IMPORTANT:"
    )
    print(
        "Repeatability is evaluated from the distribution of tau_BP "
        "across independent seeds, not from a hardcoded previous result."
    )

### Cell 14 — Save Checkpoint 3 results


In [ ]:
# ============================================================
# CELL 14 — SAVE CHECKPOINT 3 RESULTS
# ============================================================

if seed_data is not None:

    tau_seed_df.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "tau_per_seed.csv"
        ),
        index=False
    )

    seed_counts.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "seed_coverage.csv"
        ),
        index=False
    )

    repeatability_summary.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "repeatability_summary.csv"
        ),
        index=False
    )

    if "representative_df" in globals():
        representative_df.to_csv(
            os.path.join(
                OUTPUT_DIR,
                "representative_cases.csv"
            ),
            index=False
        )

    checkpoint_summary = {
        "source_file": seed_source,
        "threshold": THRESHOLD,
        "configurations_analyzed": int(
            len(repeatability_summary)
        ),
        "min_seeds_per_config": int(
            seed_counts["n_seeds"].min()
        ),
        "max_seeds_per_config": int(
            seed_counts["n_seeds"].max()
        ),
        "total_seed_observations": int(
            len(tau_seed_df)
        ),
        "censored_seed_observations": int(
            tau_seed_df["censored"].sum()
        ),
    }

    with open(
        os.path.join(
            OUTPUT_DIR,
            "checkpoint3_summary.json"
        ),
        "w"
    ) as f:
        json.dump(
            checkpoint_summary,
            f,
            indent=2
        )

    print("=" * 80)
    print("CHECKPOINT 3 FILES SAVED")
    print("=" * 80)

    for fname in sorted(
        os.listdir(OUTPUT_DIR)
    ):
        print(
            os.path.join(
                OUTPUT_DIR,
                fname
            )
        )

    print("\nSTATUS: COMPLETE")

else:

    print(
        "\nSTATUS: INCOMPLETE"
    )